In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Dijkstra's Shortest Path Algorithm

In [ ]:
import { insert, remove, elevate, HeapElement, HeapNode, DijkstraNode, heapToDot } from './Heap-Array';

In [ ]:
function shortestPath(source: string, edges: Record<string, Array<[string, number]>>): Record<string, number> {
    const distance: Record<string, number> = { [source]: 0 };
    const fringe: HeapElement[] = [];
    const node = new DijkstraNode(source);
    insert(fringe, [0, node]);
    const nodeMap: Record<string, HeapNode> = { [source]: node };
    while (fringe.length > 0) {
        const [d, u] = remove(fringe);
        const uName = (u as DijkstraNode).mValue;
        for (const [v, l] of edges[uName]) {
            const dv = distance[v];
            if (dv === undefined || d + l < dv) {
                if (dv !== undefined) {
                    elevate(fringe, nodeMap[v], d + l);
                } else {
                    const newNode = new DijkstraNode(v);
                    nodeMap[v] = newNode;
                    insert(fringe, [d + l, newNode]);
                }
                distance[v] = d + l;
            }
        }
    }
    return distance;
}

## Code to Display the Directed Graph

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";
import { display } from "tslab";

The function $\texttt{toDot}(\texttt{source}, \texttt{Edges}, \texttt{Fringe}, \texttt{Distance}, \texttt{Visited})$ takes a graph that is represented by 
its `Edges`, a set of nodes `Fringe`, and a dictionary `Distance` that has the distance of a node from the node `source`,  and set `Visited` of nodes that have already been visited.

In [ ]:
async function toDot(
    source: string,
    p: string | null,
    edges: Record<string, Array<[string, number]>>,
    fringe: HeapElement[],
    distance: Record<string, number>,
    visited: Record<string, boolean>
): Promise<void> {
    const V = new Set<string>();
    for (const x of Object.keys(edges)) {
        V.add(x);
    }
    let dot = `digraph G {\nnode [shape=record style=rounded]\nrankdir=LR;\nsize="8,5";\n`;
    for (const x of V) {
        const distanceLabel = distance[x] !== undefined ? String(distance[x]) : '';
        if (x === source) {
            dot += `  "${x}" [color=blue shape=doublecircle];\n`;
        } else if (x === p) {
            dot += `  "${x}" [label="{${x}|${distanceLabel}}" color=magenta];\n`;
        } else if (distance[x] !== undefined && fringe.some(([d, n]) => d === distance[x] && (n as DijkstraNode).mValue === x)) {
            dot += `  "${x}" [label="{${x}|${distanceLabel}}" color=red];\n`;
        } else if (visited[x]) {
            dot += `  "${x}" [label="{${x}|${distanceLabel}}" color=blue];\n`;
        } else {
            dot += `  "${x}" [label="{${x}|${distanceLabel}}"];\n`;
        }
    }
    for (const u of V) {
        for (const [v, l] of edges[u]) {
            dot += `  "${u}" -> "${v}" [label="${l}"];\n`;
        }
    }
    dot += `}`;
    const gv = await Graphviz.load();
    const svg = gv.layout(dot, "svg", "dot");
    const widthMatch = svg.match(/width="([^"]+)"/);
    const heightMatch = svg.match(/height="([^"]+)"/);
    const formattedWidth = widthMatch ? widthMatch[0].replace(/"/g, '').replace('=', ':') : '';
    const formattedHeight = heightMatch ? heightMatch[0].replace(/"/g, '').replace('=', ':') : '';
    const html = `
        <div style='${formattedWidth}; ${formattedHeight};'>
            ${svg}
        </div>
    `;
    display.html(html);
}

Below is an implementation of Dijkstra's shortest path algorithm that adds some visualization.

In [ ]:
function shortestPath(source: string, edges: Record<string, Array<[string, number]>>): Record<string, number> {
    const distance: Record<string, number> = { [source]: 0 };
    const visited: Record<string, boolean> = { [source]: true };
    const fringe: HeapElement[] = [];
    const nodeMap: Record<string, HeapNode> = {};
    const node = new DijkstraNode(source);
    nodeMap[source] = node;
    insert(fringe, [0, node]);
    while (fringe.length > 0) {
        heapToDot(fringe);
        const [d, u] = remove(fringe);
        const uName = (u as DijkstraNode).mValue;
        toDot(source, uName, edges, fringe, distance, visited);
        console.log('_'.repeat(80));
        for (const [v, l] of edges[uName]) {
            const dv = distance[v];
            if (dv === undefined || d + l < dv) {
                if (dv !== undefined) {
                    elevate(fringe, nodeMap[v], d + l);
                } else {
                    const newNode = new DijkstraNode(v);
                    nodeMap[v] = newNode;
                    insert(fringe, [d + l, newNode]);
                }
                distance[v] = d + l;
            }
        }
        visited[uName] = true;
    }
    toDot(source, null, edges, fringe, distance, visited);
    return distance;
}

## Code for Testing

In [ ]:
const a = new DijkstraNode('a');
const b = new DijkstraNode('b');
const c = new DijkstraNode('c');
const d = new DijkstraNode('d');
const e = new DijkstraNode('e');
const f = new DijkstraNode('f');
const g = new DijkstraNode('g');
const h = new DijkstraNode('h');

In [ ]:
const edges: Record<string, Array<[string, number]>> = {
    [a.mValue]: [[c.mValue, 2], [b.mValue, 9]],
    [b.mValue]: [[d.mValue, 1]],
    [c.mValue]: [[e.mValue, 5], [g.mValue, 3]],
    [d.mValue]: [[f.mValue, 2], [e.mValue, 4]],
    [e.mValue]: [[f.mValue, 1], [b.mValue, 2]],
    [f.mValue]: [[h.mValue, 5]],
    [g.mValue]: [[e.mValue, 1]],
    [h.mValue]: []
};

In [ ]:
const sp = shortestPath(a.mValue, edges);
console.log(sp);

# Crossing the Tunnel

Four persons, Alice, Britney, Charly and Daniel have to cross a tunnel.
The tunnel is so narrow, that at most two persons can cross it together.
In order to cross the tunnel, a torch is needed.  Together, they only
have a single torch.
  1. Alice   is the fastest and can cross the tunnel in 1 minute.
  2. Britney needs 2 minutes to cross the tunnel.
  3. Charly  is slower and needs 4 minutes.
  4. Daniel  is slowest and takes 5 minutes to cross the tunnel.
  
What is the fastest plan to cross the tunnel?

In [ ]:
const All = new Set(['Alice', 'Britney', 'Charly', 'Daniel', 'Torch']);
const Time: Record<string, number> = {
    'Alice': 1,
    'Britney': 2,
    'Charly': 4,
    'Daniel': 5,
    'Torch': 0
};

The function $\texttt{power}(M)$ defined below computes the power list of the set $M$, i.e. we have:
$$ \texttt{power}(M) = 2^M = \bigl\{A \mid A \subseteq M \bigr\} $$

In [ ]:
function power(M: Set<string>): Set<Set<string>> {
    if (M.size === 0) {
        return new Set([new Set()]);
    } else {
        const C = new Set(M);
        const x = C.values().next().value;
        C.delete(x);
        const P1 = power(C);
        const P2 = new Set<Set<string>>();
        for (const A of P1) {
            const newSet = new Set(A);
            newSet.add(x);
            P2.add(newSet);
        }
        return new Set([...P1, ...P2]);
    }
}

If $B$ is a set of persons, then $\texttt{duration}(B)$ is the time that this group needs to cross the tunnel.
$B$ also contains `'Torch'`.

In [ ]:
const nodeDict: Record<string, DijkstraNode> = {};
for (const S of power(All)) {
    const key = Array.from(S).sort().join(',');
    nodeDict[key] = new DijkstraNode(key);
}

In [ ]:
function duration(B: Set<string>): number {
    return Math.max(...Array.from(B).map(x => Time[x]));
}

$\texttt{left_right}(S)$ describes a crossing of the tunnel from left to right.

In [ ]:
function leftRight(S: Set<string>): Array<[DijkstraNode, number]> {
    const result: Array<[DijkstraNode, number]> = [];
    for (const B of power(S)) {
        if (B.has('Torch') && B.size >= 2 && B.size <= 3) {
            const left = new Set(S);
            for (const x of B) {
                left.delete(x);
            }
            const leftKey = Array.from(left).sort().join(',');
            result.push([nodeDict[leftKey], duration(B)]);
        }
    }
    return result;
}

$\texttt{right_left}(S)$ describes a crossing of the tunnel from right to left.

In [ ]:
function rightLeft(S: Set<string>): Array<[DijkstraNode, number]> {
    const result: Array<[DijkstraNode, number]> = [];
    const diff = new Set(All);
    for (const x of S) {
        diff.delete(x);
    }
    for (const B of power(diff)) {
        if (B.has('Torch') && B.size >= 2 && B.size <= 3) {
            const right = new Set(S);
            for (const x of B) {
                right.add(x);
            }
            const rightKey = Array.from(right).sort().join(',');
            result.push([nodeDict[rightKey], duration(B)]);
        }
    }
    return result;
}

In [ ]:
const edges: Record<string, Array<[DijkstraNode, number]>> = {};
for (const S of power(All)) {
    const key = Array.from(S).sort().join(',');
    edges[key] = [...leftRight(S), ...rightLeft(S)];
}
Object.keys(edges).length;

The function `shortest_path` is Dikstra's algorithm.  It returns both a dictionary containing the parents and a dictionary 
with the distances.

In [ ]:
function shortestPath(source: string, edges: Record<string, Array<[DijkstraNode, number]>>): [Record<string, string>, Record<string, number>] {
    const distance: Record<string, number> = { [source]: 0 };
    const parent: Record<string, string> = {};
    const fringe: HeapElement[] = [];
    const nodeMap: Record<string, HeapNode> = {};
    const node = new DijkstraNode(source);
    nodeMap[source] = node;
    insert(fringe, [0, node]);
    while (fringe.length > 0) {
        const [d, u] = remove(fringe);
        const uName = (u as DijkstraNode).mValue;
        for (const [v, l] of edges[uName]) {
            const vName = (v as DijkstraNode).mValue;
            const dv = distance[vName];
            if (dv === undefined || d + l < dv) {
                if (dv !== undefined) {
                    elevate(fringe, nodeMap[vName], d + l);
                } else {
                    const newNode = new DijkstraNode(vName);
                    nodeMap[vName] = newNode;
                    insert(fringe, [d + l, newNode]);
                }
                distance[vName] = d + l;
                parent[vName] = uName;
            }
        }
    }
    return [parent, distance];
}

In [ ]:
const start = new DijkstraNode(Array.from(All).sort().join(','));

In [ ]:
const [parent, distance] = shortestPath(start.mValue, edges);

Let us see whether the goal was reachable and how long it takes to reach the goal.

In [ ]:
const goal = nodeDict[''];
distance[goal.mValue];

Given to nodes `source` and `goal` and a dictionary containing the parent of every node, the function
`find_path` returns the path from `source` to `goal`.

In [ ]:
function findPath(source: string, goal: string, parent: Record<string, string>): string[] {
    const p = parent[goal];
    if (p === undefined) {
        return [source];
    }
    return [...findPath(source, p, parent), goal];
}

In [ ]:
const path = findPath(start.mValue, goal.mValue, parent).map(x => nodeDict[x].mValue);

In [ ]:
function printPath(): void {
    let total = 0;
    console.log("_".repeat(81));
    for (let i = 0; i < path.length; i++) {
        const left = new Set<string>(path[i].split(',').filter(Boolean));
        const right = new Set<string>([...All].filter(x => !left.has(x)));
        if (left.size === 0 || right.size === 0) {
            console.log(left, " ".repeat(25), right);
        } else {
            console.log(left, " ".repeat(30), right);
        }
        console.log("_".repeat(81));
        if (i < path.length - 1) {
            if (path[i].includes('Torch')) {
                const diff = new Set<string>([...left].filter(x => !path[i + 1].split(',').includes(x)));
                const time = duration(diff);
                total += time;
                console.log(" ".repeat(20), ">>> ", diff, ':', time, " >>>");
            } else {
                const diff = new Set<string>([...right].filter(x => path[i + 1].split(',').includes(x)));
                const time = duration(diff);
                total += time;
                console.log(" ".repeat(20), "<<< ", diff, ':', time, " <<<");
            }
            console.log("_".repeat(81));
        }
    }
    console.log('Total time:', total, 'minutes.');
}

In [ ]:
printPath();